# Data Analysis 

**LSE ID:** 250094127

**Question:** How has the average runtime of top-grossing movies changed across 2015–2025, and does it vary by gender?


This notebook reads the tidy tables from NB02 and develops three findings. It does not repeat collection or preparation work.

## 1. Setup

I load the three tables built in NB02 from the SQLite database.

In [1]:
import pandas as pd
import plotly.express as px
import sqlite3

In [2]:
conn = sqlite3.connect("../data/movies.db")
movies = pd.read_sql("SELECT * FROM movies", conn)
genres = pd.read_sql("SELECT * FROM genres", conn)
genres_movie = pd.read_sql("SELECT * FROM movie_genres", conn)



I also build one long table with **one row per movie–genre pair**, joining runtime and year onto each genre. This is the table for the genre-level views (Findings 2 and 3).

In [3]:
q_genres = pd.read_sql("""

SELECT 
    m.year,
    m.id,
    m.title,
    g.genre_name,
    m.runtime
FROM movies m
JOIN movie_genres g ON m.id = g.id
ORDER BY m.year ASC

""", conn) 

q_genres.head()

,year,id,title,genre_name,runtime
0,2015,140607,Star Wars: The Force Awakens,Adventure,136
1,2015,140607,Star Wars: The Force Awakens,Action,136
2,2015,140607,Star Wars: The Force Awakens,Science Fiction,136
3,2015,135397,Jurassic World,Adventure,124
4,2015,135397,Jurassic World,Science Fiction,124


**Note on double-counting:** a film with several genres appears once per genre in `q_genres`. That is correct when I break results down *by genre*, but it would inflate a plain per-year average. So Finding 1 (not split by genre) uses the deduplicated `movies` table, where each film appears once.

**Checking for bad runtime values.** Before computing any averages, I check for films with implausibly short runtimes — these look like bad or corrupt data from TMDB rather than real theatrical releases.

In [4]:
movies[movies["runtime"] < 45][["id", "title", "year", "runtime"]]

,id,title,year,runtime
505,1198553,Nidja's Kitchen 2,2020,4
520,1029460,The Western,2020,10
587,972402,Lost Generation,2020,25
857,1175807,Honk,2023,6
924,1462164,Jigsaw Prank Gone Wrong,2024,5
948,1292380,moi.digital untitled Volume 2,2024,18
1079,1693620,VIDEOMINUTO,2025,1


Seven films have runtimes under 45 minutes — as low as 1 minute — which isn't believable for a top-grossing theatrical release. I keep them in views that show the full distribution (the histogram and box plot below, where they still show up as outliers), but exclude them from every chart or statistic built on a **mean**, since a handful of near-zero values would drag those averages down.

In [5]:
movies_avg = movies[movies["runtime"] >= 45]
q_genres_avg = q_genres[q_genres["runtime"] >= 45]

## Finding 1 — Runtime across the years

**Does the typical top-grossing film get longer or shorter over 2015–2025?**

I first look at the full distribution per year, then reduce it to a single mean-per-year trend line.

In [6]:
fig_years = px.histogram(
    movies, x="runtime", facet_row="year", width=700, height=2000,
    labels={"runtime": "Runtime (minutes)"},
)

fig_years.update_layout(
    title=dict(
        text="Most top-grossing films run 90–140 minutes, whatever the year",
        subtitle=dict(text="Distribution of runtime (minutes) per film, faceted by year"),
    )
)
fig_years.update_xaxes(rangemode="tozero")
fig_years.update_yaxes(title_text="Number of films", rangemode="tozero")
fig_years

In [7]:
runtime_by_year = movies_avg.groupby("year")["runtime"].mean().reset_index()

fig_runtime_by_year = px.line(
    runtime_by_year, x="year", y="runtime", markers=True,
    labels={"runtime": "Mean runtime (minutes)", "year": "Year"},
)
fig_runtime_by_year.update_layout(
    title=dict(
        text="Runtimes dipped in 2020, then ran about 10 minutes longer by 2023",
        subtitle=dict(text="Mean runtime (minutes) of top-grossing films, by year"),
    )
)
fig_runtime_by_year.update_yaxes(rangemode="tozero")
fig_runtime_by_year

**Finding 1:** Runtime dipped in the pandemic year 2020 (110.9 min) and then trended upward, peaking in 2023 at 125.8 min before easing back to ~121–123 min in 2024–2025 — about 10–15 minutes longer than the 2020 low. The 2015 starting point (117.1 min) and the 2024–2025 level are fairly close, so the overall shift across the decade is modest; 2020 reads as a dip tied to that year's disrupted theatrical releases rather than the start of a longer-run trend.

## Finding 2 — Runtime differs by genre

**Which genres run long, which run short, and how spread out are they?**

I order the genres by their interquartile range (IQR = 75th − 25th percentile) so the box plot goes from least-dispersed to most-dispersed.

In [8]:
order = (
    q_genres.groupby("genre_name")["runtime"]
    .apply(lambda s: s.quantile(0.75) - s.quantile(0.25))
    .sort_values().index.tolist()
)
order



['Family',
 'Animation',
 'Documentary',
 'Horror',
 'Mystery',
 'Comedy',
 'War',
 'Romance',
 'Crime',
 'Drama',
 'Fantasy',
 'Science Fiction',
 'History',
 'Action',
 'Thriller',
 'Adventure',
 'Music',
 'Western']

In [9]:
fig_box = px.box(
    q_genres, x="runtime", y="genre_name", color="genre_name",
    points="outliers", category_orders={"genre_name": order},
    labels={"genre_name": "Genre", "runtime": "Runtime (minutes)"},
)
fig_box.update_layout(
    title=dict(
        text="Family films stick to a tight runtime; Adventure and Thriller swing far more",
        subtitle=dict(text="Runtime (minutes) distribution by genre, ordered by spread (IQR)"),
    )
)
fig_box.update_xaxes(rangemode="tozero")

In [10]:
q_genres["genre_name"].value_counts()

genre_name
Action             451
Adventure          368
Comedy             363
Drama              341
Thriller           240
Fantasy            200
Science Fiction    190
Family             185
Animation          173
Crime              151
Horror             140
Romance            107
Mystery             98
History             77
War                 42
Music               37
Western              7
Documentary          5
Name: count, dtype: int64

**Finding 2:** Among genres with enough films to judge (roughly 100+), Family runs the tightest (IQR ≈ 14 min), close to Animation (≈15 min), while Adventure and Thriller swing the most (IQR ≈ 29 min each). Western (IQR ≈ 56 min) and Documentary (≈15 min) look extreme in the plot but only have 7 and 5 films respectively, so their spread isn't reliable.

## Finding 3 — Which genres vary most over time

I average runtime **by genre and year**, plot one line per genre, then measure each genre's variation with the standard deviation of its yearly averages.

I restrict to the best-covered genres — sparse genres have too few films per year for a stable yearly average (see Limitations).


In [11]:
top_genres = q_genres_avg.groupby("genre_name")["id"].count().nlargest(4).index

runtime_genre_year = (
    q_genres_avg[q_genres_avg["genre_name"].isin(top_genres)]
    .groupby(["genre_name","year"])["runtime"].mean()
    .reset_index()
)

runtime_genre_year_plot = px.line(
    runtime_genre_year, x="year", y="runtime", color="genre_name",
    labels={"runtime": "Mean runtime (minutes)", "year": "Year", "genre_name": "Genre"},
)
runtime_genre_year_plot.update_layout(
    title=dict(
        text="Drama's average runtime swung the most across the decade; Comedy stayed steadiest",
        subtitle=dict(text="Mean runtime (minutes) by year, for the four best-sampled genres"),
    )
)
runtime_genre_year_plot.update_yaxes(rangemode="tozero")
runtime_genre_year_plot

In [12]:
variation = (
    runtime_genre_year.groupby("genre_name")["runtime"].std()
    .sort_values(ascending=False)
    .reset_index(name="runtime_std")
)
variation

,genre_name,runtime_std
0,Drama,6.936770
1,Action,6.019829
2,Adventure,5.358048
3,Comedy,3.189719


**Finding 3:** Among the four best-sampled genres, Drama's yearly average runtime varied the most (std ≈ 6.9 min) and Comedy varied the least (std ≈ 3.2 min), with Action (≈6.0 min) and Adventure (≈5.4 min) in between.

## Limitations

- **Near-zero / missing runtimes.** Seven films have implausibly short runtimes (1–25 minutes), almost certainly bad data rather than real releases (checked in Setup). I exclude them from every chart or statistic built on a mean (Finding 1's line chart, Finding 3), but keep them in the distribution views (the histogram and box plot), where they still show up as outliers.
- **Sparse genres.** Some genres have very few top-grossing films per year, so their yearly averages are unstable — Finding 3 is restricted to the four best-covered genres for this reason, and Finding 2 flags Western and Documentary (7 and 5 films) as unreliable despite their extreme-looking spread.
- **Multi-genre films.** A film contributes to each of its genres, so genre-level runtimes are not independent. This is expected for genre comparison, and is why Finding 1 uses the deduplicated film table.
- **Top-revenue selection.** The sample is only the highest-grossing films each year, so findings describe blockbusters, not all films.

## Exporting figures for the public report

I save the three figures behind Findings 1–3 as static PNGs for the project's public page (`docs/index.md`). The exploratory histogram above isn't included — it's scaffolding I used while exploring, not one of the findings.

In [13]:
import os
os.makedirs("../docs/images", exist_ok=True)

fig_runtime_by_year.write_image("../docs/images/finding1-mean-runtime-by-year.png", width=900, height=500, scale=2)
fig_box.write_image("../docs/images/finding2-runtime-by-genre-box.png", width=900, height=600, scale=2)
runtime_genre_year_plot.write_image("../docs/images/finding3-runtime-genre-year-trend.png", width=900, height=500, scale=2)

/tmp/ipykernel_50220/1769612271.py:4: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_runtime_by_year.write_image("../docs/images/finding1-mean-runtime-by-year.png", width=900, height=500, scale=2)


/tmp/ipykernel_50220/1769612271.py:5: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_box.write_image("../docs/images/finding2-runtime-by-genre-box.png", width=900, height=600, scale=2)
/tmp/ipykernel_50220/1769612271.py:6: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  runtime_genre_year_plot.write_image("../docs/images/finding3-runtime-genre-year-trend.png", width=900, height=500, scale=2)
